In [ ]:
import pandas as pd
import requests
from tqdm import tqdm

#### Download HKHS Code and Example

In [ ]:
!curl -L -o ../data/raw/B2XX00232026XXXXB0100.csv https://www.censtatd.gov.hk/en/data/stat_report/product/B2XX0023/att/B2XX00232026XXXXB0100.csv
!curl -L -o ../data/raw/index.php https://www.post.japanpost.jp/service/send/oversea/use/label/hscode/index.php

#### Fetch HKHS Category and Keyword

In [ ]:
session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/140.0.0.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;q=0.9,"
        "image/avif,image/webp,*/*;q=0.8"
    ),
    "Accept-Language": "en-US,en;q=0.9",
})

initial = session.get(
    "https://www.censtatd.gov.hk/en/index_hs_code.html",
    timeout=30,
)
initial.raise_for_status()

In [ ]:
response = session.post(
    "https://www.censtatd.gov.hk/search/index.php",
    params={
        "lang_search": "en",
        "l": "web",
        "c": "HsCode",
        "m": "chapterList",
    },
    headers={
        "Origin": "https://www.censtatd.gov.hk",
        "Referer": "https://www.censtatd.gov.hk/en/index_hs_code.html",
        "Content-Type": "application/x-www-form-urlencoded",
    },
    timeout=30,
)
response.raise_for_status()

categories = pd.DataFrame(response.json())

In [ ]:
def fetch_commodity_details(session, hs_code):
    response = session.post(
        "https://www.censtatd.gov.hk/search/index.php",
        params={
            "lang_search": "en",
            "l": "web",
            "c": "HsCode",
            "m": "searchChapterC",
        },
        data={
            "fhsCode": hs_code,
            "lang_search": "en",
            "keyword_searched": "",
        },
        headers={
            "Origin": "https://www.censtatd.gov.hk",
            "Referer": "https://www.censtatd.gov.hk/en/index_hs_code.html",
            "Content-Type": "application/x-www-form-urlencoded",
        },
        timeout=30,
    )
    response.raise_for_status()

    subcat = pd.DataFrame(response.json()["searchChapter"])
    keyword = pd.DataFrame(response.json()["ucTradeCodeKeyword"])

    return subcat, keyword

In [ ]:
hkhs = pd.read_csv(
    "../data/raw/B2XX00232026XXXXB0100.csv",
    dtype=str,
    skiprows=1,
)
subcats, kwds = [], []

for hs in tqdm(hkhs["HKHS Code (2026)"].str.zfill(8).unique()):
    try:
        subcat, kwd = fetch_commodity_details(session, hs)
        subcats.append(subcat)
        kwds.append(kwd)
    except Exception as e:
        print(f"Failed to fetch data for HS code: {hs}. Error: {e}")

subcategories = pd.concat(subcats, ignore_index=True)
keywords = pd.concat(kwds, ignore_index=True)

In [ ]:
categories.to_parquet("../data/raw/categories.parquet", index=False)
subcategories.to_parquet("../data/raw/subcategories.parquet", index=False)
keywords.to_parquet("../data/raw/keywords.parquet", index=False)